# Phase 1 + 3 — fine-tune M_pop and run identification (GPU)

Switch the Colab runtime to a **GPU** first (Runtime → Change runtime type).
Set your Minitaur checkpoint in `configs/default.yaml:model.base_model`.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!git clone https://github.com/YifeiCAO/sro-minitaur-transfer.git
%cd sro-minitaur-transfer
!pip -q install -e .
!pip -q install -r requirements-model.txt

### HuggingFace login (required)

The base model is `marcelbinz/Llama-3.1-Minitaur-8B` (merged, not gated as of writing) — you just need to be **logged in** to HF. Create a read token at https://huggingface.co/settings/tokens, add it as an `HF_TOKEN` secret in Colab (🔑 left sidebar), and run the cell below. (If the model page ever shows an "Agree and access" banner, click it once.)

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

### Phase 1 — population fine-tune (loss masked to `<<response>>` tokens), then freeze.

In [ ]:
!python scripts/finetune_mpop.py --config configs/default.yaml --subset starting_subset --out results/mpop

### Phase 3 — identification sanity. The floor (z ignored) should sit near chance = 1/K; that validates the scoring/identification plumbing end-to-end. The transfer-z path turns on once Phase 2 is trained.

In [ ]:
!python scripts/run_identification.py --mpop results/mpop --target two_stage_decision --K 10

### Phase 2 (next) — train the person-encoder `E_A` + injection on TRAIN subjects, then re-run identification and the Phase-4 NLL contrast (real vs floor vs shuffled-z) on heldout. See `src/sro_transfer/model/transfer_model.py` for the training loop scaffold.